# 07 — Experimental Design: sample size, power and blocking (Task 6)

Task 6 does not require running an experiment. This notebook supplies the **numbers** behind the written evaluation of a
Completely Randomised Design (CRD) and a Randomised Complete Block Design (RCBD) for a retention-offer pilot
(`docs/task6_experimental_design.md`): how many customers each design needs, which outcome is realistically detectable, and how
much blocking helps.

It uses the customer table built at the 9 Sep 2011 cutoff as a stand-in for the customer base at the time of a future pilot.
The repurchase rate and spend distribution are taken as planning values; a real pilot would recompute them at its own start date.

**Run notebook 00 first.**

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import norm

customers = pd.read_csv('../../data/processed/customer_table.csv')
ALPHA, POWER = 0.05, 0.80
Z_A, Z_B = norm.ppf(1 - ALPHA / 2), norm.ppf(POWER)

p0 = customers['Repurchase'].mean()
spend = customers['FutureSpend']
print(f'Customers: {len(customers)} | 90-day repurchase rate: {p0:.3f}')
print(f'90-day spend: mean GBP {spend.mean():.0f}, SD GBP {spend.std():.0f}, '
      f'coefficient of variation {spend.std() / spend.mean():.1f}, median GBP {spend.median():.0f}')

Customers: 5253 | 90-day repurchase rate: 0.434
90-day spend: mean GBP 530, SD GBP 3148, coefficient of variation 5.9, median GBP 0


## 1. CRD: customers per arm to detect a rise in the repurchase rate

Two-sided two-proportion test, 5% significance, 80% power, equal arms. The effect is the offer's **incremental** effect on the
repurchase rate (percentage points above the control group's rate).

In [2]:
def n_per_arm(p1, p2):
    pbar = (p1 + p2) / 2
    num = Z_A * np.sqrt(2 * pbar * (1 - pbar)) + Z_B * np.sqrt(p1 * (1 - p1) + p2 * (1 - p2))
    return int(np.ceil((num / (p2 - p1)) ** 2))

crd = pd.DataFrame({'uplift (pp)': [3, 5, 8, 10]})
crd['n per arm (CRD)'] = [n_per_arm(p0, p0 + d / 100) for d in crd['uplift (pp)']]
crd['total (2 arms)'] = 2 * crd['n per arm (CRD)']
crd['share of current base'] = (crd['total (2 arms)'] / len(customers)).round(2)
crd

,uplift (pp),n per arm (CRD),total (2 arms),share of current base
0,3,4314,8628,1.64
1,5,1558,3116,0.59
2,8,611,1222,0.23
3,10,391,782,0.15


## 2. RCBD: how much does blocking help?

Candidate blocks must be known **before** the pilot starts. Recency tertile x Frequency tertile (9 blocks) uses only
pre-cutoff behaviour. Blocking removes the between-block variation from the error term, so the required sample shrinks by
roughly the share of outcome variance the blocks explain (R², used here as an approximation).

In [3]:
c = customers.copy()
c['R'] = pd.qcut(c['Recency'], 3, labels=['recent', 'mid', 'old'])
c['F'] = pd.qcut(c['Frequency'].rank(method='first'), 3, labels=['low', 'mid', 'high'])
c['block'] = 'R-' + c['R'].astype(str) + ' / F-' + c['F'].astype(str)

def r2_blocks(y, g):
    fitted = y.groupby(g).transform('mean')
    return 1 - ((y - fitted) ** 2).sum() / ((y - y.mean()) ** 2).sum()

r2 = {'Repurchase (binary)': r2_blocks(c['Repurchase'], c['block']),
      'log(1 + spend)': r2_blocks(np.log1p(c['FutureSpend']), c['block']),
      'Spend (GBP)': r2_blocks(c['FutureSpend'], c['block'])}
print('Share of outcome variance explained by the 9 blocks:', {k: round(v, 3) for k, v in r2.items()})

blocks = c.groupby('block').agg(customers=('Repurchase', 'size'), repurchase_rate=('Repurchase', 'mean'),
                                mean_spend=('FutureSpend', 'mean')).sort_values('repurchase_rate')
blocks.round(3)

Share of outcome variance explained by the 9 blocks: {'Repurchase (binary)': np.float64(0.234), 'log(1 + spend)': np.float64(0.273), 'Spend (GBP)': np.float64(0.039)}


,customers,repurchase_rate,mean_spend
block,,,
R-old / F-low,1035,0.143,73.572
R-mid / F-low,481,0.225,103.084
R-old / F-mid,583,0.262,135.379
R-old / F-high,122,0.270,204.206
R-mid / F-mid,716,0.385,209.016
R-recent / F-low,235,0.515,327.672
R-recent / F-mid,452,0.540,453.329
R-mid / F-high,552,0.583,470.771
R-recent / F-high,1077,0.811,1731.572


In [4]:
rcbd = crd[['uplift (pp)', 'n per arm (CRD)']].copy()
rcbd['n per arm (RCBD, approx.)'] = np.ceil(rcbd['n per arm (CRD)'] * (1 - r2['Repurchase (binary)'])).astype(int)
rcbd['saving'] = (1 - rcbd['n per arm (RCBD, approx.)'] / rcbd['n per arm (CRD)']).round(2)
rcbd

,uplift (pp),n per arm (CRD),"n per arm (RCBD, approx.)",saving
0,3,4314,3303,0.23
1,5,1558,1193,0.23
2,8,611,468,0.23
3,10,391,300,0.23


## 3. Is spend a realistic primary outcome?

Customers per arm to detect a relative rise in **mean 90-day spend** (two-sample comparison of means, normal approximation).

In [5]:
rows = []
for rel in [0.10, 0.20, 0.50]:
    delta = rel * spend.mean()
    n_crd = 2 * (Z_A + Z_B) ** 2 * spend.var() / delta ** 2
    rows.append({'relative uplift in mean spend': rel, 'n per arm (CRD)': int(np.ceil(n_crd)),
                 'n per arm (RCBD, approx.)': int(np.ceil(n_crd * (1 - r2['Spend (GBP)'])))})
pd.DataFrame(rows)

,relative uplift in mean spend,n per arm (CRD),"n per arm (RCBD, approx.)"
0,0.1,55298,53120
1,0.2,13825,13280
2,0.5,2212,2125


## 4. Minimum detectable effect for pilots the retailer could actually run

In [6]:
def mde_pp(n_arm, p=p0):
    # smallest uplift (pp) detectable with n_arm customers per arm (search on a fine grid)
    for d in np.arange(0.001, 0.30, 0.001):
        if n_per_arm(p, p + d) <= n_arm:
            return round(100 * d, 1)

top3 = c[c['block'].isin(blocks.index[-3:])]
# (customers per arm in the smaller arm, baseline repurchase rate of that population).
# Using the smaller arm for an unequal split is conservative.
scenarios = {'Whole active base, 50/50': (len(customers) // 2, p0),
             'Whole active base, 80% treated / 20% control': (int(0.2 * len(customers)), p0),
             'Top 3 value blocks only, 50/50': (len(top3) // 2, top3['Repurchase'].mean())}
pd.DataFrame({'customers per arm (smaller arm)': {k: v[0] for k, v in scenarios.items()},
              'baseline repurchase rate': {k: round(v[1], 3) for k, v in scenarios.items()},
              'minimum detectable uplift, CRD (pp)': {k: mde_pp(*v) for k, v in scenarios.items()}})

,customers per arm (smaller arm),baseline repurchase rate,"minimum detectable uplift, CRD (pp)"
"Whole active base, 50/50",2626,0.434,3.9
"Whole active base, 80% treated / 20% control",1050,0.434,6.1
"Top 3 value blocks only, 50/50",1040,0.691,5.6


## Findings

*Numbers from the current data; re-check if notebook 00 or `src/` changes.*

* **Repurchase, not spend, must be the primary outcome.** 90-day spend has a coefficient of variation of 5.9 (mean £530, SD
  £3,148, median £0), so detecting even a 10% rise in mean spend needs about 55,000 customers per arm, ten times the whole
  customer base. A 20% rise needs about 13,800 per arm. Spend can be a secondary outcome only.
* **A 5-point rise in the repurchase rate is detectable.** From a 43.4% baseline, a CRD needs 1,558 customers per arm (3,116 in
  total, 59% of the current base); 8 points needs 611 per arm; 3 points needs 4,314 per arm, more than the base can supply.
* **Blocking on recency x frequency tiers saves about 23% of the sample.** The nine blocks explain 23.4% of the variance in
  repurchase (repurchase rates range from 14% to 81% across blocks), so an RCBD needs roughly 1,193 per arm for a 5-point effect.
  They explain only 3.9% of raw spend variance, because a few very large buyers dominate it, so blocking does not rescue
  spend as an outcome.
* **What realistic pilots can detect:** splitting the whole base 50/50 detects about 3.9 points; an 80/20 treated/control split
  about 6.1 points (the control arm limits it); a 50/50 pilot on the three highest-value blocks only (2,081 customers, baseline
  69%) about 5.6 points.
* The blocking saving is an approximation (variance explained by the block means); a real analysis would estimate it with a
  logistic model including block effects.
